**Entrenamiento de clasificador de ojos**

In [5]:
from mediapipe_model_maker import image_classifier
from inference.classifier_run import create_clasifier

# importar datos
TRAIN_PATH = "MRL-FL3D/train"
VALID_PATH = "MRL-FL3D/val"
TEST_PATH = "MRL-FL3D/test"
data_train = image_classifier.Dataset.from_folder(TRAIN_PATH)
data_valid = image_classifier.Dataset.from_folder(VALID_PATH)
data_test = image_classifier.Dataset.from_folder(TEST_PATH)

INFO:tensorflow:Load image with size: 78016, num_label: 2, labels: awake, sleepy.


INFO:tensorflow:Load image with size: 78016, num_label: 2, labels: awake, sleepy.


INFO:tensorflow:Load image with size: 16980, num_label: 2, labels: awake, sleepy.


INFO:tensorflow:Load image with size: 16980, num_label: 2, labels: awake, sleepy.


INFO:tensorflow:Load image with size: 16981, num_label: 2, labels: awake, sleepy.


INFO:tensorflow:Load image with size: 16981, num_label: 2, labels: awake, sleepy.


In [3]:
# Entrenamiento
MODEL_DIR="./eye-classifier-b"
SAVE_NAME = "en0_eye_bplus.tflite"
QUANTIZED_NAME = "en0_eye_bplus.f16.tflite"
EPOCHS = 8
BATCH_SIZE = 32
model_options = image_classifier.ModelOptions(dropout_rate=0.1)
hparams = image_classifier.HParams(
    learning_rate=2e-04,
    export_dir=MODEL_DIR,
    shuffle=True,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
)
model_arch = image_classifier.SupportedModels.EFFICIENTNET_LITE0
options = image_classifier.ImageClassifierOptions(supported_model=model_arch, hparams=hparams, model_options=model_options)

classifier = image_classifier.ImageClassifier.create(train_data=data_train, validation_data=data_valid, options=options)

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 keras_layer_1 (KerasLayer)  (None, 1280)              11837936  
                                                                 
 dropout_1 (Dropout)         (None, 1280)              0         
                                                                 
 dense_1 (Dense)             (None, 2)                 2562      
                                                                 
Total params: 11840498 (45.17 MB)
Trainable params: 2562 (10.01 KB)
Non-trainable params: 11837936 (45.16 MB)
_________________________________________________________________
None


ValueError: Received incompatible tensor with shape (1, 1, 32, 16) when attempting to restore variable with shape (1, 1, 32, 24) and name efficientnet-lite4/blocks_0/conv2d/kernel:0.

In [3]:
# Precision en datos de prueba
loss, accuracy = classifier.evaluate(data_test)
print(f"Dataset: {TRAIN_PATH}\nPrecision: {accuracy*100}%")

INFO:tensorflow:Use customized resize method bilinear


INFO:tensorflow:Use customized resize method bilinear


531/531 [==============================] - 251s 471ms/step - loss: 0.2891 - accuracy: 0.9562
Dataset: MRL-FL3D/train
Precision: 95.62452435493469%


In [4]:
from mediapipe_model_maker import quantization

# Guardar y cuantizar
classifier.export_model(model_name=SAVE_NAME)
config = quantization.QuantizationConfig.for_float16()
classifier.export_model(model_name=QUANTIZED_NAME, quantization_config=config)

INFO:tensorflow:Assets written to: /tmp/tmp3hyln3oi/saved_model/assets


INFO:tensorflow:Assets written to: /tmp/tmp3hyln3oi/saved_model/assets
2026-03-13 19:55:25.384795: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-03-13 19:55:25.384855: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-03-13 19:55:25.389077: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp3hyln3oi/saved_model
2026-03-13 19:55:25.411788: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-03-13 19:55:25.411836: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp3hyln3oi/saved_model
2026-03-13 19:55:25.442544: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2026-03-13 19:55:25.454660: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-03-13 19:55:26.013227: I tensorflow/cc/saved_model/

INFO:tensorflow:TensorFlow Lite model exported successfully to: ./eye-classifier-b/en0_eye_b.tflite


INFO:tensorflow:TensorFlow Lite model exported successfully to: ./eye-classifier-b/en0_eye_b.tflite


INFO:tensorflow:Assets written to: /tmp/tmp1zhupv2y/saved_model/assets


INFO:tensorflow:Assets written to: /tmp/tmp1zhupv2y/saved_model/assets
2026-03-13 19:55:55.920840: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-03-13 19:55:55.920889: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-03-13 19:55:55.921077: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp1zhupv2y/saved_model
2026-03-13 19:55:55.930865: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-03-13 19:55:55.930902: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp1zhupv2y/saved_model
2026-03-13 19:55:55.963811: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-03-13 19:55:56.445832: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmp1zhupv2y/saved_model
2026-03-13 19:55:56.780408: I ten

INFO:tensorflow:TensorFlow Lite model exported successfully to: ./eye-classifier-b/en0_eye_b.f16.tflite


INFO:tensorflow:TensorFlow Lite model exported successfully to: ./eye-classifier-b/en0_eye_b.f16.tflite


In [7]:
# Validacion en FL3D cropped
FL3D_VAL = "FL3D-eyesbig"
data_extval = image_classifier.Dataset.from_folder(FL3D_VAL)


INFO:tensorflow:Load image with size: 4161, num_label: 2, labels: awake, sleepy.


INFO:tensorflow:Load image with size: 4161, num_label: 2, labels: awake, sleepy.


In [8]:
loss, accuracy = classifier.evaluate(data_extval)
print(f"Dataset: {FL3D_VAL}\nPrecision: {accuracy*100}%")


INFO:tensorflow:Use customized resize method bilinear


INFO:tensorflow:Use customized resize method bilinear


131/131 [==============================] - 63s 478ms/step - loss: 0.4036 - accuracy: 0.8409
Dataset: FL3D-eyesbig
Precision: 84.0903639793396%
